In [35]:
import pandas as pd
import numpy as np

merged = pd.read_csv("merged_mycotoxins.csv", low_memory=False)

# Clean LOQ column
merged["loq"] = pd.to_numeric(merged["loq"], errors="coerce")

# Overall LOQ stats
print("=== OVERALL LOQ DISTRIBUTION ===")
print(merged["loq"].describe(percentiles=[.05, .10, .25, .50, .75, .90, .95]))
print(f"\nN with LOQ: {merged['loq'].notna().sum():,}")
print(f"N missing LOQ: {merged['loq'].isna().sum():,}")

# Per contaminant
print("\n=== LOQ PER CONTAMINANT ===")
contam_loq = (
    merged.groupby("contaminant")["loq"]
    .agg(["count", "min", "median", "mean", "max"])
    .round(3)
    .sort_values("median")
)
print(contam_loq.to_string())

# Per country
print("\n=== LOQ PER COUNTRY ===")
country_loq = (
    merged.groupby("samplecountry")["loq"]
    .agg(["count", "min", "median", "mean", "max"])
    .round(3)
)
print(country_loq.to_string())

# Per country x contaminant
print("\n=== LOQ PER COUNTRY x CONTAMINANT ===")
country_contam_loq = (
    merged.groupby(["samplecountry", "contaminant"])["loq"]
    .agg(["count", "min", "median", "max"])
    .round(3)
)
print(country_contam_loq.to_string())

# Value distribution relative to key LOQ thresholds
loq_median = merged["loq"].median()
loq_p25    = merged["loq"].quantile(0.25)
loq_p75    = merged["loq"].quantile(0.75)
loq_max    = merged["loq"].max()

print(f"\n=== DETECTION RATES AT DIFFERENT LOQ THRESHOLDS ===")
print(f"{'Threshold':<20} {'Value':>10} {'N detected':>12} {'% detected':>12}")
for label, val in [
    ("25th percentile", loq_p25),
    ("Median (50th)",   loq_median),
    ("75th percentile", loq_p75),
    ("Max LOQ",         loq_max),
]:
    n_det = (merged["value"].fillna(0) > val).sum()
    pct   = n_det / len(merged) * 100
    print(f"{label:<20} {val:>10.3f} {n_det:>12,} {pct:>11.1f}%")

=== OVERALL LOQ DISTRIBUTION ===
count    12243.000000
mean        12.688544
std         17.268173
min          0.090000
5%           0.220000
10%          1.550000
25%          2.680000
50%          5.000000
75%         10.000000
90%         50.000000
95%         50.000000
max        100.000000
Name: loq, dtype: float64

N with LOQ: 12,243
N missing LOQ: 0

=== LOQ PER CONTAMINANT ===
                              count    min  median    mean     max
contaminant                                                       
enniatin a + a1                 517   0.09    0.09   0.090    0.09
enniatin b + b1                 517   0.22    0.22   0.220    0.22
ergotaminine                    517   1.67    1.67   1.670    1.67
ergometrinine                   517   1.91    1.91   1.910    1.91
ergocryptinine                  517   2.68    2.68   2.680    2.68
ergocorninine                   517   2.76    2.76   2.760    2.76
ergosinine                      517   3.85    3.85   3.850    3.85
alternar

In [36]:
print(merged.groupby(["harvest_year", "samplecountry"]).size().unstack().fillna(0).astype(int))

samplecountry   cz   de    dk   nl
harvest_year                      
2019.0         130   35   765  575
2020.0         220  521  1005  872
2021.0         385  586  1190  700
2022.0         435  503  1270  652
2023.0         425  266  1380  328


In [37]:

merged["loq"] = pd.to_numeric(merged["loq"], errors="coerce")
merged["value"] = pd.to_numeric(merged["value"], errors="coerce")

countries = sorted(merged["samplecountry"].dropna().unique())
contaminants = sorted(merged["contaminant"].dropna().unique())

# -- Per country x contaminant LOQ frequency table -------------------------
rows = []
for country in countries:
    for contam in contaminants:
        sub = merged[
            (merged["samplecountry"] == country) &
            (merged["contaminant"] == contam)
        ]
        if len(sub) == 0:
            continue

        loq_counts = sub["loq"].dropna().value_counts().sort_index()
        for loq_val, count in loq_counts.items():
            n_above  = (sub["value"].fillna(0) > loq_val).sum()
            n_total  = len(sub)
            rows.append({
                "country":       country,
                "contaminant":   contam,
                "loq_value":     loq_val,
                "n_samples":     count,
                "pct_of_group":  round(count / n_total * 100, 1),
                "n_above_loq":   n_above,
                "pct_above_loq": round(n_above / n_total * 100, 1),
            })

freq_df = pd.DataFrame(rows)

# -- Print -----------------------------------------------------------------
for country in countries:
    print(f"\n{'='*70}")
    print(f"  COUNTRY: {country.upper()}")
    print(f"{'='*70}")
    
    for contam in contaminants:
        sub = freq_df[
            (freq_df["country"] == country) &
            (freq_df["contaminant"] == contam)
        ]
        if len(sub) == 0:
            continue

        n_total = sub["n_samples"].sum()
        print(f"\n  {contam}  (n={n_total:,})")
        print(f"  {'LOQ':>8}  {'N':>6}  {'% of group':>11}  {'N > LOQ':>8}  {'% > LOQ':>8}")
        print(f"  {'─'*50}")
        for _, r in sub.iterrows():
            print(f"  {r['loq_value']:>8.3f}  {r['n_samples']:>6,}  "
                  f"{r['pct_of_group']:>10.1f}%  {r['n_above_loq']:>8,}  "
                  f"{r['pct_above_loq']:>7.1f}%")

# -- Summary table: median & max LOQ per country x contaminant ------------
print(f"\n\n{'='*90}")
print("SUMMARY — median & max LOQ and detection rates per country x contaminant")
print(f"{'='*90}")
print(f"{'Country':<8} {'Contaminant':<35} {'LOQ median':>10} {'LOQ max':>10} "
      f"{'% > median':>11} {'% > max':>9}")
print("─" * 90)

for country in countries:
    for contam in contaminants:
        sub = merged[
            (merged["samplecountry"] == country) &
            (merged["contaminant"] == contam)
        ]
        if len(sub) == 0:
            continue
        loq_median = sub["loq"].median()
        loq_max    = sub["loq"].max()
        val        = sub["value"].fillna(0)
        pct_median = (val > loq_median).mean() * 100
        pct_max    = (val > loq_max).mean() * 100
        print(f"{country:<8} {contam:<35} {loq_median:>10.3f} {loq_max:>10.3f} "
              f"{pct_median:>10.1f}% {pct_max:>8.1f}%")
    print()

# -- Save ------------------------------------------------------------------
freq_df.to_csv("loq_frequency_table_by_country.csv", index=False)
print("\nSaved -> loq_frequency_table_by_country.csv")


  COUNTRY: CZ

  alternariol  (n=71)
       LOQ       N   % of group   N > LOQ   % > LOQ
  ──────────────────────────────────────────────────
     4.600      71       100.0%         5      7.0%

  beauvericin  (n=248)
       LOQ       N   % of group   N > LOQ   % > LOQ
  ──────────────────────────────────────────────────
     1.550      71        28.6%       156     62.9%
     5.000     177        71.4%        60     24.2%

  enniatin a  (n=177)
       LOQ       N   % of group   N > LOQ   % > LOQ
  ──────────────────────────────────────────────────
     5.000     177       100.0%         8      4.5%

  enniatin a + a1  (n=71)
       LOQ       N   % of group   N > LOQ   % > LOQ
  ──────────────────────────────────────────────────
     0.090      71       100.0%        59     83.1%

  enniatin a1  (n=177)
       LOQ       N   % of group   N > LOQ   % > LOQ
  ──────────────────────────────────────────────────
     5.000     177       100.0%        47     26.6%

  enniatin b  (n=177)
    

In [38]:
loq_all = pd.concat([
    merged["loq"].dropna(),
], ignore_index=True)

loq_median = loq_all.median()
loq_max    = loq_all.max()
print(f"  Median LOQ : {loq_median}")
print(f"  Max LOQ    : {loq_max}")


  Median LOQ : 5.0
  Max LOQ    : 100.0


In [39]:
rows = []

for contam in sorted(merged["contaminant"].dropna().unique()):

    sub = merged[merged["contaminant"] == contam].copy()

    if len(sub) == 0:
        continue

    # original
    original_loq = sub["loq"]

    # harmonized floor LOQ
    sub["blanket5_loq"] = sub["loq"].clip(lower=5)

    original_detected = (
        sub["value"] > original_loq
    ).sum()

    blanket_detected = (
        sub["value"] > sub["blanket5_loq"]
    ).sum()

    loss = original_detected - blanket_detected

    rows.append({
        "contaminant": contam,
        "n_samples": len(sub),

        "original_detected": original_detected,
        "blanket5_detected": blanket_detected,

        "loss": loss,
        "loss_pct":
            100 * loss / max(original_detected,1),

        "median_original_loq":
            original_loq.median(),

        "median_harmonized_loq":
            sub["blanket5_loq"].median(),

        "max_original_loq":
            original_loq.max()
    })


blanket5_summary = pd.DataFrame(rows)

blanket5_summary = blanket5_summary.sort_values(
    "loss_pct",
    ascending=False
)

print(blanket5_summary.round(2))

                     contaminant  n_samples  original_detected  \
6                enniatin a + a1        517                436   
9                enniatin b + b1        517                407   
16                  ergotaminine        517                 19   
4                    beauvericin       1676                174   
11                 ergocorninine        517                 10   
15                    ergosinine        517                 14   
1                    alternariol        993                 35   
7                    enniatin a1       1158                193   
10                   enniatin b1       1161                450   
8                     enniatin b       1164                538   
0                      altenuene        125                  0   
2   alternariol monomethyl ether        442                 13   
5                     enniatin a       1124                 38   
3                   altertoxin i         16                  0   
12        

In [40]:
for contam in sorted(merged["contaminant"].dropna().unique()):

    sub = merged[merged["contaminant"] == contam]

    median_loq = sub["loq"].median()

    newly_censored = (
        (sub["value"] > sub["loq"]) &
        (sub["value"] <= median_loq)
    ).sum()

    print(contam, newly_censored)

altenuene 0
alternariol 0
alternariol monomethyl ether 0
altertoxin i 0
beauvericin 53
enniatin a 6
enniatin a + a1 0
enniatin a1 25
enniatin b 14
enniatin b + b1 0
enniatin b1 19
ergocorninine 0
ergocristinine 0
ergocryptinine 0
ergometrinine 0
ergosinine 0
ergotaminine 0
tentoxin 1
tenuazonic acid 0


In [41]:
summary["blanket5_newly_censored_pct"] = (
    100 * summary["blanket_loss_pct"] / summary["n_samples"]
)

summary["median_newly_censored_pct"] = (
    100 * summary["median_loss_pct"] / summary["n_samples"]
)

summary["max_newly_censored_pct"] = (
    100 * summary["max_loss_pct"] / summary["n_samples"]
)

print(summary["blanket5_newly_censored_pct"])
print(summary["median_newly_censored_pct"])
print(summary["max_newly_censored_pct"])

2      0.000000
17   -62.893082
5     -3.980146
1      0.575457
7     -2.192453
10    -0.363671
4     -0.788687
8     -0.223559
0      0.000000
6      5.146132
3      0.000000
9      2.566308
11     1.934236
13     0.000000
12     0.000000
14     0.000000
15     1.381597
16     2.036038
18     0.000000
Name: blanket5_newly_censored_pct, dtype: float64


2      0.000000
17    62.893082
5      1.404757
1      0.000000
7      1.118598
10     0.363671
4      1.748827
8      0.223559
0      0.000000
6      0.000000
3      0.000000
9      0.000000
11     0.000000
13     0.000000
12     0.000000
14     0.000000
15     0.000000
16     0.000000
18     0.000000
Name: median_newly_censored_pct, dtype: float64
2     22.624434
17    62.893082
5      8.662671
1      8.056395
7      6.890565
10     4.766006
4      2.571805
8      3.529044
0      0.000000
6      0.000000
3      0.000000
9      0.000000
11     0.000000
13     0.000000
12     0.000000
14     0.000000
15     0.000000
16     0.000000
18     0.000000
Name: max_newly_censored_pct, dtype: float64


In [42]:
# Original detection using measured LOQ
original_detected = (
    merged["value"] > merged["loq"]
).sum()


# Median contaminant LOQ harmonization
median_loq = (
    merged.groupby("contaminant")["loq"]
    .transform("median")
)

median_detected = (
    merged["value"] > median_loq
).sum()


# Floor harmonization: LOQ cannot be lower than 5
floor5_loq = merged["loq"].clip(lower=5)

floor5_detected = (
    merged["value"] > floor5_loq
).sum()


# Maximum LOQ harmonization
max_loq = (
    merged.groupby("contaminant")["loq"]
    .transform("max")
)

max_detected = (
    merged["value"] > max_loq
).sum()


print(f"Original LOQ:       {original_detected}")
print(f"LOQ floor = 5:      {floor5_detected}")
print(f"Median LOQ:         {median_detected}")
print(f"Maximum LOQ:        {max_detected}")

Original LOQ:       2362
LOQ floor = 5:      2165
Median LOQ:         2246
Maximum LOQ:        1584


In [34]:
merged["floor5_loq"] = merged["loq"].clip(lower=5)

country_summary = (
    merged.groupby("samplecountry")
    .apply(lambda x: pd.Series({
        "original_detected":
            (x["value"] > x["loq"]).sum(),

        "floor5_detected":
            (x["value"] > x["floor5_loq"]).sum(),

        "loss_pct":
            100 * (
                (x["value"] > x["loq"]).sum()
                -
                (x["value"] > x["floor5_loq"]).sum()
            )
            /
            max((x["value"] > x["loq"]).sum(),1)
    }))
)

print(country_summary)

C:\Users\homme015\AppData\Local\Temp\ipykernel_25228\3334919629.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged["floor5_loq"] = merged["loq"].clip(lower=5)


               original_detected  floor5_detected   loss_pct
samplecountry                                               
cz                         467.0            439.0   5.995717
de                         303.0            216.0  28.712871
dk                        1356.0           1274.0   6.047198
nl                         236.0            236.0   0.000000


In [48]:

all_tables = []
summary = []

# ============================================================
# Frequency distribution per contaminant
# ============================================================

for contam in sorted(merged["contaminant"].dropna().unique()):

    sub = merged.loc[
        merged["contaminant"] == contam,
        "loq"
    ].dropna()

    if len(sub) == 0:
        continue

    median_loq = sub.median()

    freq = (
        sub.value_counts()
           .sort_index()
           .rename_axis("LOQ")
           .reset_index(name="Count")
    )

    freq["Percent"] = (
        100 * freq["Count"] / freq["Count"].sum()
    )

    freq["Cumulative Count"] = freq["Count"].cumsum()

    freq["Cumulative Percent"] = (
        100 * freq["Cumulative Count"] / freq["Count"].sum()
    )

    freq["Median LOQ"] = median_loq

    freq["Relative to median"] = freq["LOQ"].apply(
        lambda x:
            "Below median" if x < median_loq
            else "Median" if x == median_loq
            else "Above median"
    )

    freq.insert(0, "Contaminant", contam)

    all_tables.append(freq)

    summary.append({
        "Contaminant": contam,
        "N": len(sub),
        "Median LOQ": median_loq,
        "Mode LOQ": freq.loc[freq["Count"].idxmax(), "LOQ"],
        "Unique LOQs": len(freq)
    })

# ============================================================
# Combine everything
# ============================================================

frequency_table = pd.concat(all_tables, ignore_index=True)

print("\nLOQ FREQUENCY TABLE")
print("=" * 120)
print(frequency_table.round(2).to_string(index=False))

print("\n\nSUMMARY")
summary = pd.DataFrame(summary)
print(summary.to_string(index=False))

# Optional: save for inspection in Excel
frequency_table.to_csv("loq_frequency_table.csv", index=False)
summary.to_csv("loq_summary.csv", index=False)


LOQ FREQUENCY TABLE
                 Contaminant    LOQ  Count  Percent  Cumulative Count  Cumulative Percent  Median LOQ Relative to median
                   altenuene   8.00     71    56.80                71               56.80        8.00             Median
                   altenuene  50.00     54    43.20               125              100.00        8.00       Above median
                 alternariol   4.00     68     6.85                68                6.85        4.60       Below median
                 alternariol   4.60    517    52.06               585               58.91        4.60             Median
                 alternariol   8.00      3     0.30               588               59.21        4.60       Above median
                 alternariol  20.00    405    40.79               993              100.00        4.60       Above median
alternariol monomethyl ether   2.00     71    16.06                71               16.06        5.00       Below median
alternariol